# Baremetal LLM: quick demo

Train a tiny char-level LM and generate text.

[GitHub](https://github.com/Rashal10/baremetal-llm)

In [ ]:
!pip install -q git+https://github.com/Rashal10/baremetal-llm.git

In [ ]:
import torch
import torch.optim as optim
from baremetal_llm.foundation import CharDataLoader, CharTokenizer, TinyLM, train_step
from baremetal_llm.utils.paths import data_path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

data = CharDataLoader(str(data_path()), ctx_len=32)
model = TinyLM(vocab=256, ctx_len=32, n_layers=2, n_heads=2, dim=64).to(device)
opt = optim.AdamW(model.parameters(), lr=3e-3)

for step in range(40):
    x, y = data.get_batch("train", 8, device)
    loss = train_step(model, x, y, opt)
    if (step + 1) % 10 == 0:
        print(f"step {step + 1} loss={loss:.4f}")

In [ ]:
tok = CharTokenizer()
ids = tok.encode("Transformers ").unsqueeze(0).to(device)
out = model.generate(ids, max_tokens=50)
print(tok.decode(out[0]))